# Week 7, Day 6: Forge: Week 7 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w007/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", np.linalg.svd(np.array([[3.0, 0.0], [0.0, -2.0]]), compute_uv=False))
print("2.", np.linalg.matrix_rank(np.outer([1, 2, 3], [4, 5])))
lam = np.array([5.0, 3.0, 1.0, 1.0])
print("3.", lam.sum() ** 2 / np.sum(lam ** 2))

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(2014)
N, C, T, K, dt = 80, 8, 50, 30, 0.02
t = np.linspace(0, 1, T)
theta = np.arange(C) * 2 * np.pi / C
env = np.sin(np.pi * t)
z = np.zeros((3, C, T))
for c in range(C):
    phase = 2 * np.pi * 0.8 * t + theta[c]
    z[0, c], z[1, c], z[2, c] = env * np.cos(phase), env * np.sin(phase), t
mix = rng.normal(size=(N, 3))
gain = rng.uniform(0.3, 3, N)
rate = np.clip(gain[:, None, None] * (15 + 12 * np.einsum("nk,kct->nct", mix, z)), 0, None)
counts = rng.poisson(rate[..., None] * dt, size=(N, C, T, K))
del z, mix, rate                                  # in a real experiment you never see these

def to_matrix(R):
    n, c, tt = R.shape
    return R.transpose(1, 2, 0).reshape(c * tt, n)
print(counts.shape)

### Exercise: Part A: the population's spectrum

Average over trials to get `psth` in Hz, soft-normalise each neuron by its range plus 5 Hz, and build the centred data matrix `X`. From its SVD compute `var_explained`, the number of components needed for 90% (`n90`), and the participation ratio `pr`. Plot the scree plot. How many components would you guess carry signal?

In [ ]:
psth = ...
span = ...
M = to_matrix(psth / span[:, None, None])
X = ...
U, s, Vt = np.linalg.svd(X, full_matrices=False)
var_explained = ...
n90 = ...
pr = ...

plt.figure(figsize=(5, 3)); plt.semilogy(np.arange(1, 31), var_explained[:30], "o")
plt.xlabel("component"); plt.ylabel("variance explained"); plt.show()
print("90% needs", n90, "components; participation ratio", round(pr, 2))

In [ ]:
import numpy as np
def test_psth():
    "psth is the trial-averaged rate in Hz, shape (80, 8, 50)"
    assert psth.shape == (N, C, T) and np.allclose(psth, counts.mean(axis=-1) / dt)

def test_X():
    "X is the centred, soft-normalised data matrix"
    span = psth.max(axis=(1, 2)) - psth.min(axis=(1, 2)) + 5
    M = to_matrix(psth / span[:, None, None])
    assert np.allclose(X, M - M.mean(axis=0))

def test_summaries():
    "var_explained sums to 1, n90 is the components needed for 90%, pr is the participation ratio"
    assert np.isclose(var_explained.sum(), 1)
    assert n90 == int(np.searchsorted(np.cumsum(var_explained), 0.9) + 1)
    assert np.isclose(pr, 1 / np.sum(var_explained ** 2))

test_psth()
test_X()
test_summaries()
print("All checks passed")

### Exercise: Part B: choose k by cross-validation over trials

Split the trials into even (`psA`) and odd (`psB`) halves and average each. Soft-normalise both with the spans of half A, giving data matrices `A` and `B`. For k from 1 to 20, rebuild A from its top k components (adding its mean back) and record the mean squared difference from B in `cv_err`. The best k, `k_best`, is where the error is smallest. Why does this not keep falling as k grows?

In [ ]:
psA = counts[..., ::2].mean(axis=-1) / dt
psB = counts[..., 1::2].mean(axis=-1) / dt
spanA = ...
A = ...
B = ...
muA = A.mean(axis=0)
UA, sA, VtA = np.linalg.svd(A - muA, full_matrices=False)
cv_err = []
for k in range(1, 21):
    ...
k_best = ...

plt.figure(figsize=(5, 3)); plt.plot(np.arange(1, 21), cv_err, "o-"); plt.axvline(k_best, ls="--", color="0.5")
plt.xlabel("components kept from half A"); plt.ylabel("error against half B"); plt.show()
print("cross-validated choice: k =", k_best)

In [ ]:
import numpy as np
def test_halves():
    "A and B are soft-normalised data matrices from even and odd trials, both using A's spans"
    spanA = psA.max(axis=(1, 2)) - psA.min(axis=(1, 2)) + 5
    assert np.allclose(A, to_matrix(psA / spanA[:, None, None]))
    assert np.allclose(B, to_matrix(psB / spanA[:, None, None]))

def test_err():
    "cv_err[k-1] is the mean squared difference between A's rank-k reconstruction and B"
    assert len(cv_err) == 20
    muA = A.mean(axis=0)
    UA, sA, VtA = np.linalg.svd(A - muA, full_matrices=False)
    assert np.isclose(cv_err[4], np.mean((UA[:, :5] * sA[:5] @ VtA[:5] + muA - B) ** 2))

def test_k():
    "the cross-validated choice is 3 components"
    assert k_best == 3

test_halves()
test_err()
test_k()
print("All checks passed")

### Exercise: Part C: decode single trials

Decode reach direction from the spike counts in one 20 ms bin (bin 25, mid-movement). Use trials 0 to 19 for training and 20 to 29 for testing. With a nearest-centroid decoder (assign each test trial to the condition whose mean training response is closest), compute the accuracy using all 80 raw counts (`acc_raw`) and using counts scaled by `span` and projected onto the top 3 PCs from Part A, `Vt[:3]` (`acc_pc`). Chance is 1 in 8.

In [ ]:
bin_counts = counts[:, :, 25, :]                    # (80, 8, 30)
train, test = np.arange(20), np.arange(20, 30)
cent_raw = ...                                      # (8, 80): mean training counts per condition

def nearest_centroid(cents, x):
    return int(np.argmin(np.sum((cents - x) ** 2, axis=1)))

P = Vt[:3]                                          # (3, 80)
to_pc = lambda x: (x / span) @ P.T                  # scale like the PCA data, then project
cent_pc = to_pc(cent_raw)
hits_raw, hits_pc = 0, 0
for c in range(C):
    for k in test:
        x = bin_counts[:, c, k]
        ...
acc_raw, acc_pc = hits_raw / 80, hits_pc / 80
print(f"accuracy: raw counts {acc_raw:.2f}, top 3 PCs {acc_pc:.2f}, chance {1 / C:.2f}")

In [ ]:
import numpy as np
def test_centroids():
    "centroids are mean counts per condition over the 20 training trials, shape (8, 80)"
    assert cent_raw.shape == (C, N)
    assert np.allclose(cent_raw, bin_counts[:, :, :20].mean(axis=-1).T)

def test_acc():
    "accuracies are fractions between 0 and 1, from the 80 test trials"
    assert 0 <= acc_raw <= 1 and 0 <= acc_pc <= 1
    assert np.isclose(acc_raw * 80, round(acc_raw * 80))

def test_better():
    "projecting onto the top 3 PCs decodes better than all 80 raw counts"
    assert acc_pc > acc_raw

test_centroids()
test_acc()
test_better()
print("All checks passed")